# 04 — Purchase Orders

Simulates inventory drawdown from sales and triggers replenishment POs.
Requires master data and sales tables to exist.

In [ ]:
%run ./00_helpers

In [ ]:
RANDOM_SEED      = 42
SCHEMA_NAME      = "rockline"
START_DATE       = "2022-01-01"
END_DATE         = ""
VAT_RATE         = 0.20
MAX_PO_LINES     = 15
WRITE_MODE       = "overwrite"
PO_ID_OFFSET     = 0

In [ ]:
import random
from datetime import datetime, date, timedelta
from decimal import Decimal
from collections import defaultdict

rng  = random.Random(RANDOM_SEED)
_NOW = datetime.utcnow()

start_dt = date.fromisoformat(START_DATE)
end_dt   = date.fromisoformat(END_DATE) if END_DATE else _NOW.date() - timedelta(days=1)
print(f"Purchase generation window: {start_dt} to {end_dt}")

## Load Reference Data

In [ ]:
products_pd  = spark.table(f"{SCHEMA_NAME}.dim_product").toPandas()
suppliers_pd = spark.table(f"{SCHEMA_NAME}.dim_supplier").toPandas()
branches_pd  = spark.table(f"{SCHEMA_NAME}.dim_branch").toPandas()
employees_pd = spark.table(f"{SCHEMA_NAME}.dim_employee").toPandas()
snapshot_pd  = spark.table(f"{SCHEMA_NAME}.fact_inventory_snapshot").toPandas()

# Purchasing officers at HQ (branch_id=1)
buyers = employees_pd[employees_pd["job_title"].isin(
    ["Purchasing Officer", "Head of Procurement"]
)]["employee_id"].tolist()
if not buyers:
    buyers = [int(employees_pd.iloc[0]["employee_id"])]

# Product lookup: product_id -> row
prod_by_id = {int(r["product_id"]): r for _, r in products_pd.iterrows()}

# Supplier lookup: supplier_id -> lead_time_days
sup_lead = {int(r["supplier_id"]): int(r["lead_time_days"]) for _, r in suppliers_pd.iterrows()}

# Initial stock state from opening snapshot
stock = {}   # (product_id, branch_id) -> current_qty (float)
reorder_pt = {}
max_lvl    = {}
for _, row in snapshot_pd.iterrows():
    key = (int(row["product_id"]), int(row["branch_id"]))
    stock[key]     = float(row["quantity_on_hand"])
    reorder_pt[key] = float(row["reorder_point"])
    max_lvl[key]   = float(row["max_stock_level"])

print(f"Loaded inventory state for {len(stock)} product-branch pairs")

## Simulate Inventory Drawdown & Replenishment

In [ ]:
# Load daily sales deductions grouped by date
sales_pd = spark.sql(f'''
    SELECT
        CAST(so.order_date AS DATE) AS order_date,
        sol.product_id,
        so.branch_id,
        SUM(sol.quantity) AS qty_sold
    FROM {SCHEMA_NAME}.fact_sales_order so
    JOIN {SCHEMA_NAME}.fact_sales_order_line sol
      ON so.sales_order_id = sol.sales_order_id
    WHERE so.order_status != "cancelled"
      AND CAST(so.order_date AS DATE) BETWEEN "{start_dt}" AND "{end_dt}"
    GROUP BY CAST(so.order_date AS DATE), sol.product_id, so.branch_id
''').toPandas()

sales_by_date = defaultdict(list)
for _, row in sales_pd.iterrows():
    sales_by_date[row["order_date"]].append(
        (int(row["product_id"]), int(row["branch_id"]), float(row["qty_sold"]))
    )

# Pending replenishment: (supplier_id, branch_id, iso_week) -> list of (product_id, qty)
pending_pos = defaultdict(list)

def iso_week_key(d):
    iso = d.isocalendar()
    return f"{iso[0]}-W{iso[1]:02d}"

# Simulate day by day
expected_deliveries = defaultdict(list)  # deliver_date -> list of (product_id, branch_id, qty, cost)

days = date_spine(start_dt, end_dt)
for day in days:
    # Apply expected deliveries for this day
    for (pid, bid, qty, cost) in expected_deliveries.get(day, []):
        key = (pid, bid)
        stock[key] = stock.get(key, 0) + qty

    # Deduct sales
    for (pid, bid, qty_sold) in sales_by_date.get(day, []):
        key = (pid, bid)
        stock[key] = max(0.0, stock.get(key, 0) - qty_sold)

    # Check reorder triggers
    for key, qty in stock.items():
        pid, bid = key
        if qty < reorder_pt.get(key, 0):
            prod = prod_by_id.get(pid)
            if prod is None:
                continue
            sup_id   = int(prod["supplier_id"])
            week_key = iso_week_key(day)
            order_qty = max_lvl.get(key, reorder_pt.get(key, 10) * 5) - qty
            order_qty = max(1.0, round(order_qty, 3))
            pending_pos[(sup_id, bid, week_key)].append((pid, order_qty, float(prod["standard_cost_gbp"])))
            # Optimistically update stock to avoid repeated triggers this week
            stock[key] = max_lvl.get(key, order_qty)
            deliver_on = business_day_offset(day, sup_lead.get(sup_id, 5))
            if deliver_on <= end_dt:
                expected_deliveries[deliver_on].append((pid, bid, order_qty, float(prod["standard_cost_gbp"])))

print(f"Simulated {len(days)} business days, generated {len(pending_pos)} PO batches")

## Build Purchase Orders

In [ ]:
po_rows   = []
pol_rows  = []
mov_rows  = []
po_id     = PO_ID_OFFSET + 1
pol_id    = PO_ID_OFFSET * MAX_PO_LINES + 1
mov_id    = 1000000 + PO_ID_OFFSET  # offset to avoid collision with opening_balance movements

# Sort PO batches by (branch, supplier, week) for determinism
for (sup_id, bid, week_str), lines in sorted(pending_pos.items()):
    year, week = int(week_str.split("-W")[0]), int(week_str.split("-W")[1])
    # Monday of that ISO week
    order_date = date.fromisocalendar(year, week, 1)
    if order_date < start_dt:
        order_date = start_dt
    if order_date > end_dt:
        continue

    sup_row   = suppliers_pd[suppliers_pd["supplier_id"] == sup_id]
    lead      = int(sup_row["lead_time_days"].iloc[0]) if len(sup_row) else 5
    exp_del   = business_day_offset(order_date, lead)
    actual_del = exp_del if exp_del <= end_dt else None
    status = "fully_received" if actual_del else rng.choice(["submitted", "acknowledged"])
    buyer  = rng.choice(buyers)

    subtotal = Decimal("0.00")
    line_num = 0
    for (pid, qty, cost) in lines[:MAX_PO_LINES]:
        line_num += 1
        d_qty   = Decimal(str(round(qty, 3)))
        d_cost  = Decimal(str(round(cost, 4)))
        line_net = Decimal(str(round(qty * cost, 2)))
        line_vat = Decimal(str(round(float(line_net) * VAT_RATE, 2)))
        rcv_qty  = d_qty if actual_del else Decimal("0.000")
        pol_rows.append((
            pol_id, po_id, line_num, pid,
            d_qty, rcv_qty, d_cost,
            line_net, line_vat, line_net + line_vat,
            _NOW,
        ))
        subtotal += line_net
        pol_id   += 1

        if actual_del:
            mov_rows.append((
                mov_id, pid, bid,
                datetime.combine(actual_del, datetime.min.time()),
                "purchase_receipt", po_id, "PURCHASE_ORDER",
                d_qty, d_cost, _NOW,
            ))
            mov_id += 1

    vat_total = Decimal(str(round(float(subtotal) * VAT_RATE, 2)))
    po_rows.append((
        po_id, f"PO-{order_date.year}-{po_id:07d}",
        sup_id, bid, buyer,
        order_date, exp_del, actual_del,
        status,
        subtotal, vat_total, subtotal + vat_total,
        _NOW,
    ))
    po_id += 1

print(f"Built {len(po_rows)} purchase orders, {len(pol_rows)} lines, {len(mov_rows)} receipt movements")

## Write Tables

In [ ]:
po_df = to_spark_df(po_rows, SCHEMA_FACT_PURCHASE_ORDER)
po_df.write.format("delta").mode(WRITE_MODE).saveAsTable(f"{SCHEMA_NAME}.fact_purchase_order")
print(f"fact_purchase_order: {po_df.count()} rows")

pol_df = to_spark_df(pol_rows, SCHEMA_FACT_PURCHASE_ORDER_LINE)
pol_df.write.format("delta").mode(WRITE_MODE).saveAsTable(f"{SCHEMA_NAME}.fact_purchase_order_line")
print(f"fact_purchase_order_line: {pol_df.count()} rows")

if mov_rows:
    mov_df = to_spark_df(mov_rows, SCHEMA_FACT_INVENTORY_MOVEMENT)
    mov_df.write.format("delta").mode("append").saveAsTable(f"{SCHEMA_NAME}.fact_inventory_movement")
    print(f"fact_inventory_movement (receipts appended): {len(mov_rows)} rows")